# EUDR Coffee Risk Analytics — Phase 1 (MVP)

**Goal:** for a set of coffee plots (polygons), detect tree-cover loss after the EUDR cutoff
(31 Dec 2020) using Hansen Global Forest Change, flag each plot, and show it on an interactive map.

**Scope of Phase 1:** load polygons -> overlay Hansen loss -> per-plot stats -> map.
Later phases add the official JRC EUDR forest baseline, GLAD/RADD near-real-time alerts,
a proper risk-scoring model, a land-legality layer, and a DDS-ready export.

**Cost:** $0. All datasets are open Earth Engine assets. No paid API.


## Setup

You need a (free, non-commercial) Google Earth Engine account and a GEE Cloud project.
Easiest path: run this notebook in **Google Colab**, where auth is a one-click flow.

1. Sign up: https://earthengine.google.com (register a Cloud project).
2. Run the install cell once.
3. On first run, uncomment `ee.Authenticate()` and set your project id in `ee.Initialize(...)`.


In [ ]:
# Run once (Colab). Skip if these are already installed in your local env.
!pip install -q earthengine-api geemap geopandas

In [ ]:
import ee
import geemap
import pandas as pd

# First run only: uncomment to authenticate (opens a browser / prints a link).
# ee.Authenticate()

ee.Initialize(project="YOUR_GEE_PROJECT")  # <-- set your GEE Cloud project id

## Step 1 — Load plots

Sample polygons near Di Linh, Lâm Đồng. **Replace `plots_geojson` with your real plot
boundaries** (export from QGIS / GPS as GeoJSON). One plot is intentionally > 4 ha to show
the EUDR point-vs-polygon rule.

In [ ]:
# Sample coffee plots (Di Linh, Lam Dong). Swap in your real polygons here.
plots_geojson = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"plot_id": "DL-001"},
         "geometry": {"type": "Polygon", "coordinates": [[
             [108.0700, 11.5800], [108.0712, 11.5800],
             [108.0712, 11.5811], [108.0700, 11.5811], [108.0700, 11.5800]]]}},
        {"type": "Feature", "properties": {"plot_id": "DL-002"},
         "geometry": {"type": "Polygon", "coordinates": [[
             [108.0750, 11.5850], [108.07706, 11.5850],
             [108.07706, 11.58703], [108.0750, 11.58703], [108.0750, 11.5850]]]}},
        {"type": "Feature", "properties": {"plot_id": "DL-003"},
         "geometry": {"type": "Polygon", "coordinates": [[
             [108.0660, 11.5755], [108.06747, 11.5755],
             [108.06747, 11.57694], [108.0660, 11.57694], [108.0660, 11.5755]]]}},
    ],
}

plots = geemap.geojson_to_ee(plots_geojson)
print("Loaded plots:", plots.size().getInfo())

## Step 2 — Area + EUDR geometry rule

EUDR allows a single point for plots < 4 ha; plots >= 4 ha need a polygon.

In [ ]:
def add_area(f):
    area_ha = f.geometry().area(maxError=1).divide(10000)
    eudr_geom = ee.Algorithms.If(area_ha.lt(4), "point allowed", "polygon required")
    return f.set({"area_ha": area_ha, "eudr_geom": eudr_geom})

plots = plots.map(add_area)

## Step 3 — Hansen post-2020 loss layer

`lossyear` encodes year of loss (value 21 = 2021). EUDR cutoff is 31 Dec 2020, so we keep
loss in 2021+ that occurred on land that was forest (canopy >= threshold) in 2000.

> NOTE: Hansen GFC is versioned ~yearly. Check the GEE catalog for the latest asset id and
> update the string below. Hansen detects *change*; the official "forest at 2020" layer for
> a real DDS is the JRC EUDR baseline, added in a later phase.

In [ ]:
gfc = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")  # <-- verify latest version

FOREST_THRESHOLD = 30  # % canopy cover that counts as "forest"
forest2000 = gfc.select("treecover2000").gte(FOREST_THRESHOLD)
lossyear = gfc.select("lossyear")

# Loss from 2021 onwards, on land that was forest in 2000.
loss_post2020 = gfc.select("loss").And(lossyear.gte(21)).And(forest2000)

## Step 4 — Per-plot stats

Sum the lost area inside each plot, compute % of plot affected, assign a simple risk tier.
(The tiering here is a placeholder; Phase 2 replaces it with a real scoring model.)

In [ ]:
pixel_area = ee.Image.pixelArea()
loss_area_img = loss_post2020.multiply(pixel_area).rename("loss_area")

def plot_stats(f):
    stats = loss_area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=f.geometry(),
        scale=30,
        maxPixels=int(1e9),
    )
    lost_m2 = ee.Number(stats.get("loss_area"))
    plot_m2 = f.geometry().area(maxError=1)
    lost_pct = lost_m2.divide(plot_m2).multiply(100)
    tier = ee.Algorithms.If(
        lost_pct.gt(1), "high",
        ee.Algorithms.If(lost_pct.gt(0), "standard", "low"))
    return f.set({
        "lost_ha": lost_m2.divide(10000),
        "loss_pct": lost_pct,
        "risk_tier": tier,
    })

results = plots.map(plot_stats)

df = geemap.ee_to_df(results)
cols = ["plot_id", "area_ha", "eudr_geom", "lost_ha", "loss_pct", "risk_tier"]
df = df[[c for c in cols if c in df.columns]].round(3)
df

## Step 5 — Interactive map

Satellite basemap, Hansen post-2020 loss in red, plots outlined by status
(green = clean, red = flagged).

In [ ]:
Map = geemap.Map()
Map.centerObject(plots, 14)
Map.add_basemap("SATELLITE")

# Hansen post-2020 loss
Map.addLayer(loss_post2020.selfMask(), {"palette": ["red"]}, "Forest loss 2021+")

# Plots split by flag
flagged = results.filter(ee.Filter.gt("loss_pct", 0))
clean = results.filter(ee.Filter.eq("loss_pct", 0))
Map.addLayer(clean.style(color="00FF00", fillColor="00FF0033"), {}, "Plots - clean")
Map.addLayer(flagged.style(color="FF0000", fillColor="FF000055"), {}, "Plots - flagged")

Map

## Next steps

- **Real data:** replace the sample GeoJSON with your Lâm Đồng plots.
- **JRC EUDR baseline:** add the official forest-at-2020 layer for DDS-grade results.
- **GLAD / RADD alerts:** layer near-real-time deforestation alerts for recent loss.
- **Risk scoring (Phase 2):** combine loss %, proximity to forest, alert density, land-legality.
- **Land legality:** overlay Vietnam forest-zoning / protected-area maps.
- **DDS export (Phase 3):** emit geolocation + risk fields in a Due Diligence Statement structure.
